In [1]:
## conda env!!!

In [1]:
from config import CLASSES_TO_USE, ORGANISMS_TO_USE, PER_ORGANISM_PER_CLASS_ALLELES, RESULT_COLUMNS

#rewrite classes to use to only I class because it's only class I method
METHOD = 'puffin'
ORGANISMS_TO_USE = ['Human']
ALLELES = [item  for organism_to_use in ORGANISMS_TO_USE 
                 for class_to_use in CLASSES_TO_USE 
                 for item in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]]
ALLELES

['HLA_A_02_01',
 'HLA_B_27_05',
 'HLA_A_03_01',
 'HLA_B_07_02',
 'HLA_DRB1_01_01',
 'HLA_DRB1_04_01',
 'HLA_DRB1_04_05']

In [2]:
# importing the predict module
from mhcnames import normalize_allele_name## ONLY FOR LINUX

import pandas as pd
import os
import sys

In [20]:
# # Read datasets
# DECOY_KEY = 'x1'

# DATA_PATH = "/home/denklewer/Data/"
# INPUT_DATA_PATH = DATA_PATH + 'input/'
# TARGET_PATH_TO_PREDICTIONS = DATA_PATH + 'predictions/'+ METHOD+'/'
# per_allele_val_df = {
#     allele_name :pd.read_csv("{}df_{}_{}_val_for_classification.csv".format(INPUT_DATA_PATH,allele_name,DECOY_KEY), index_col=0) for allele_name in ALLELES
# }

In [5]:
def defineClass(allele_name):
    for organism_to_use in ORGANISMS_TO_USE:
        for class_to_use in CLASSES_TO_USE:
            if allele_name in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]:
                return class_to_use

In [23]:
for allele_name in ALLELES:
    per_allele_val_df[allele_name]['source_sequence_name'] =  'seq' + per_allele_val_df[allele_name].index.astype(str).values
#     per_allele_val_df[allele_name]['length'] = per_allele_val_df[allele_name]['peptide'].str.len()
    
#     #Filter for mhcnuggets <=15 length for class 1 and <=30 for class 2
#     if defineClass(allele_name) == 'I':
#         per_allele_val_df[allele_name] = per_allele_val_df[allele_name][per_allele_val_df[allele_name]['length'] <= 15]
#     else:
#         per_allele_val_df[allele_name] = per_allele_val_df[allele_name][per_allele_val_df[allele_name]['length'] <= 30]

In [7]:
per_allele_val_df[ALLELES[0]].head()

,peptide,binder
ID,,
0,KLEPIWNEVGLE,1
1,ASFVEVVSLW,1
2,GSLDKTSP,1
3,KLDDTVYLRDIA,1
4,SFPFWNLTG,1


In [9]:

TARGET_ALLELES = dict()

## HARDCODE
for allele_name in ALLELES:
    if allele_name == 'H2_d class II':
        target_allele_name = 'H2_IAd'
    elif allele_name == 'H2_b class II':
        target_allele_name = 'H2_IAb'
    elif allele_name == 'HLA_DRB1_01_01':
        target_allele_name = 'DRB1_0101'
    elif allele_name == 'HLA_DRB1_04_01':
        target_allele_name = 'DRB1_0401'
    elif allele_name == 'HLA_DRB1_04_05':
        target_allele_name = 'DRB1_0405'    
    else:
        target_allele_name = allele_name
    TARGET_ALLELES[allele_name] = (target_allele_name)


TARGET_ALLELES

{'HLA_A_02_01': 'HLA_A_02_01',
 'HLA_A_03_01': 'HLA_A_03_01',
 'HLA_B_07_02': 'HLA_B_07_02',
 'HLA_B_27_05': 'HLA_B_27_05',
 'HLA_DRB1_01_01': 'DRB1_0101',
 'HLA_DRB1_04_01': 'DRB1_0401',
 'HLA_DRB1_04_05': 'DRB1_0405'}

In [11]:
## create tempfiles for puffin
TEMPDIR = 'TMP/'
for allele_name in ALLELES:
    if defineClass(allele_name) == 'I':
        target_allele = normalize_allele_name(allele_name).replace('*','')
    else:
        target_allele = TARGET_ALLELES[allele_name]
    per_allele_val_df[allele_name]['mhc'] = target_allele
    per_allele_val_df[allele_name].to_csv("{}puffin_data_{}_{}".format(TEMPDIR,allele_name,DECOY_KEY), sep='\t',index=False, header=False)


In [12]:
import os


In [13]:
for class_to_use in CLASSES_TO_USE:
    for organism_to_use in ORGANISMS_TO_USE:
        for allele_name in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]:
            if class_to_use == 'I':
                class_arg = 1
            else:
                class_arg = 2
            input_file = 'TMP/puffin_data_{}_x1'.format(allele_name)
            print(input_file)
            output_folder = 'TMP/outdir_puffin/{}'.format(allele_name)
            command = 'python $PUFFIN_HOME/preprocess.py -i {} -o {} -c {}'.format(input_file,output_folder, class_arg)
            os.system(command)
#     ret,output = commands.getstatutoutput( command )
    
   # !python $PUFFIN_HOME/preprocess.py -i $input_file -o $output_folder -c 1


TMP/puffin_data_HLA_A_02_01_x1
TMP/puffin_data_HLA_B_27_05_x1
TMP/puffin_data_HLA_A_03_01_x1
TMP/puffin_data_HLA_B_07_02_x1
TMP/puffin_data_HLA_DRB1_01_01_x1
TMP/puffin_data_HLA_DRB1_04_01_x1
TMP/puffin_data_HLA_DRB1_04_05_x1


In [14]:
# Calculate scores
for class_to_use in CLASSES_TO_USE:
    for organism_to_use in ORGANISMS_TO_USE:
        for allele_name in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]:
            if class_to_use == 'I':
                class_arg = 1
            else:
                class_arg = 2

            output_folder = 'TMP/outdir_puffin/{}'.format(allele_name)
            command = 'python $PUFFIN_HOME/score.py -o {} -c {}'.format(output_folder, class_arg)
            os.system(command)

In [26]:
#now predictions are in /TMP/outdir_puffin/allele/PUFFIN.combined
#read, add binding label and peptide
per_allele_predictions = {
    allele_name :pd.read_csv('TMP/outdir_puffin/{}/PUFFIN.combined'.format(allele_name), sep='\t') for allele_name in ALLELES
}
for allele_name in ALLELES:
    per_allele_predictions[allele_name]['peptide'] = per_allele_val_df[allele_name]['peptide']
    per_allele_predictions[allele_name]['binder'] = per_allele_val_df[allele_name]['binder']
    per_allele_predictions[allele_name]['source_sequence_name'] = per_allele_val_df[allele_name]['source_sequence_name']
    per_allele_predictions[allele_name] = per_allele_predictions[allele_name].rename(columns={"mean_pred":"score"})
per_allele_predictions[allele_name].head()

,mean_pred,epistemic_var,aleatoric_var,binding_likelihood,peptide,binder,source_sequence_name
0,0.189695,0.004591,0.027741,0.078313,MHSAEKVPLQNPPIKWAEGH,1,seq0
1,0.269564,0.010096,0.025187,0.162719,VRSITPNEKQPE,1,seq1
2,0.437026,0.004635,0.022507,0.530287,RAASLCLPRYSFRRLD,1,seq2
3,0.327942,0.008607,0.019132,0.240027,PGPGQLTLRLRACGLNF,1,seq3
4,0.400963,0.001609,0.057019,0.458870,KELQVVAWDILLNIIERLLQQLQT,1,seq4


#### write to files

In [29]:
# for decoy_key in DECOY_PREFIXES:
for allele_name, total_df in per_allele_predictions.items():
    total_df.to_csv("{}{}_predictions_{}_{}".format(TARGET_PATH_TO_PREDICTIONS,METHOD,allele_name,DECOY_KEY))